# Fetch Encounter Ranking (How-to-use)

1. **Set Parameters:**  
   Use the widgets to select `encounter_id`, `debug` mode, and `pages`.

2. **Run Notebook:**  
   Just hit "Run All" after setting parameters. The following happens:
   - Imports dependencies and settings.
   - Fetches and combines rankings for the selected encounter and pages.
   - Saves results as JSON.
   - Output are printed and saved for analysis.

3. **Debug Mode:**  
   If `debug` is `True`, uses a sample payload.

## Import Dependencies, Job Settings, Schemas, and Helpers

1. Imports all required Python libraries
2. Add and import custom job settings, schema definitions, and helper functions.

In [0]:
import asyncio
import json
from datetime import datetime
from httpx import AsyncClient
from typing import Annotated, Callable, ClassVar

import pandas as pd
import numpy as np
from pydantic import (
    BaseModel, 
    field_validator, 
    Field,
    model_serializer
)
from unittest.mock import patch, AsyncMock, MagicMock

In [0]:
from schemas import GraphQLQuery
from settings import fflogs_settings as settings

## Seting this notebook's parameters

In [0]:
def define_task_widget():
    dbutils.widgets.dropdown(
        name="encounter_id", 
        defaultValue="101", 
        choices=["101", "102", "103", "104", "105"], 
        label="Encounter ID to fetch rankings from"
    )
    dbutils.widgets.dropdown(
        name="debug",
        defaultValue="True",
        choices=["True", "False"],
        label="Debug mode to run notebook-wise."
    )
    dbutils.widgets.dropdown(
        name="pages",
        defaultValue="1",
        choices=["1", "5", "10"],
        label="Total number of pages to fetch"
    )

## Encounter Ranking Query & Payload

1. Presents the GraphQL query used to fetch encounter rankings.
2. A sample of the expected successful response payload (also usable later in `debug` mode to test flow)

In [0]:
class EncounterRankingQuery(GraphQLQuery):
    QUERY: ClassVar[str] = """
    query GetEncounterID($id: Int!, $page: Int) {
      worldData {
        encounter(id: $id) { 
          id
          name
          zone {
            id
            name
          }
          characterRankings(page: $page)
        }
      }
    }
  """
    id: int
    page: int = 1

In [0]:
ENCOUNTER_RANKING_JSON = {
    "worldData": {
        "encounter": {
            "id": 1,
            "name": "Futures Rewritten (Ultimate)",
            "zone": {
                "id": 56,
                "name": "Ultimates (Dawntrail)"
            },
            "characterRankings": {
                "rankings": [
                    {
                        "name": "Lyra Starweaver",
                        "spec": "WhiteMage",
                        "amount": 12453.67,
                        "aDPS": 11980.23,
                        "startTime": 1717200000000,
                        "duration": 485000,
                        "lodestoneID": 12345678,
                        "guild": {
                            "id": 16,
                            "name": "Please Be Careful"
                        },
                        "server": {
                            "id": 8,
                            "name": "Gilgamesh"
                        }
                    }
                ]
            }
        }
    }
}

## Send Query to Endpoint & Batch Results

1. Send a GraphQL query to the endpoint asynchronously
2. Batch multiple queries with `asyncio.gather` concurrency to combine multiple pages.
3. Write results to volume

In [0]:
async def query_graphql(
    client: AsyncClient,
    query: GraphQLQuery, 
    access_token: str
) -> dict:
    """Executes a GraphQL query against the configured endpoint asynchronously.

    Args:
        client (AsyncClient): The httpx async client to use for the request.
        query (GraphQLQuery): The GraphQL query object to execute.
        access_token (str): Bearer token used to authenticate the request.

    Returns:
        dict: The 'data' field of the GraphQL response.

    Raises:
        ValueError: If the GraphQL response contains errors.
        httpx.HTTPStatusError: If the HTTP request fails.
    """
    _FFLOGS_QUERY_ENDPOINT = "https://www.fflogs.com/api/v2/client"

    response = await client.post(
        _FFLOGS_QUERY_ENDPOINT,
        json=query.model_dump(),
        headers={
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json",
        },
    )
    response.raise_for_status()
    
    return response.json()["data"]

In [0]:
async def batch_query_graphql(
    client: AsyncClient,
    encounter_id: int,
    access_token: str,
    pages: int = 10,
) -> dict:
    """Fetches and aggregates encounter rankings across multiple pages asynchronously.

    Args:
        client (AsyncClient): The httpx async client to use for requests.
        encounter_id (int): The ID of the encounter to query.
        access_token (str): Bearer token used to authenticate the request.
        pages (int, optional): Number of pages to fetch. Defaults to 10.

    Returns:
        dict: The aggregated GraphQL response with all rankings combined.

    Raises:
        ValueError: If any GraphQL response contains errors.
        httpx.HTTPStatusError: If any HTTP request fails.
    """
    queries = [
        EncounterRankingQuery(id=encounter_id, page=page) 
        for page in range(1, pages + 1)
    ]
    results = await asyncio.gather(*[
        query_graphql(client, query, access_token)
        for query in queries
    ])

    all_rankings = []
    for data in results:
        all_rankings.extend(
            data["worldData"]["encounter"]["characterRankings"]["rankings"]
        )

    results[0]["worldData"]["encounter"]["characterRankings"]["rankings"] = all_rankings
    
    return results[0]

In [0]:
def write_json_to_volume(
    data: dict, 
    volume_dir: str,
    file_name: str
) -> None:
    now = datetime.now().strftime("%Y%m%d_%H%M%S")

    with open(f"{volume_dir}/{file_name}_{now}.json", "w") as f:
        json.dump(data, f, indent=4)
        

## Orchestrate All Tasks & Debug/Test Mode

1. Coordinates the entire workflow
2. Running test functions when debug mode is enabled.

In [0]:
async def query_encounter_to_volume() -> None:
    define_task_widget()

    encounter_id = int(dbutils.widgets.get("encounter_id"))
    pages = int(dbutils.widgets.get("pages"))

    async with AsyncClient() as client:
        result = await batch_query_graphql(
            client=client,
            encounter_id=encounter_id,
            access_token=settings.fflogs_access_token,
            pages=pages
        )

    write_json_to_volume(
        data=result,
        volume_dir=settings.volume_dir,
        file_name=f"encounter_{encounter_id}"
    )
    
    print(f"Encounter {encounter_id} has been written to volume {settings.volume_dir}")
    print(json.dumps(result, indent=4))

In [0]:
async def test_query_encounter_to_volume() -> None:
    mock_client = AsyncMock(spec=AsyncClient)
    with (
        patch("httpx.AsyncClient.__aenter__", return_value=mock_client),
        patch("httpx.AsyncClient.__aexit__", return_value=False),
        patch("__main__.batch_query_graphql",
        new=AsyncMock(return_value=ENCOUNTER_RANKING_JSON)) as mock_batch,
        patch("__main__.write_json_to_volume") as mock_write,
    ):
        await query_encounter_to_volume()

        mock_batch.assert_awaited_once_with(
            client=mock_client,
            encounter_id=int(dbutils.widgets.get("encounter_id")),
            access_token=settings.fflogs_access_token,
            pages=int(dbutils.widgets.get("pages"))
        )
        mock_write.assert_called_once_with(
            data=ENCOUNTER_RANKING_JSON,
            volume_dir=settings.volume_dir,
            file_name=f"encounter_{dbutils.widgets.get('encounter_id')}"
        )
        print("✓ test_query_encounter_to_volume passed")

In [0]:
if dbutils.widgets.get("debug") == "True":
    await test_query_encounter_to_volume()
else:
    await query_encounter_to_volume()